In [4]:
import numpy as np
import os
import json
import random
from typing import List, Dict, Tuple

# --- CRITICAL GLOBAL PARAMETER: WHITE GAUSSIAN NOISE MAGNITUDE ---
# Set to 2.0 for maximum difficulty
NOISE_MAGNITUDE = 2.0 

# --- SIGNAL PARAMETERS ---
DURATION = 0.2 # seconds
SAMPLE_RATE = 40000 # Hz. Nyquist is 20,000 Hz.
MIN_FREQ = 500
MAX_FREQ = 20000

# --- POSITIVE VARIATION SETTING ---
# Frequency variation for positive examples is tightly set to 2% (0.02)
DEFAULT_FREQ_RANGE_PCT = 0.1
DEFAULT_AMP_RANGE_PCT = 0.2

# -------------------------------------------
# Helper functions
# -------------------------------------------

def generate_component(f0, a0, freq_range_pct, amp_range_pct):
    """
    Generate a single sine component with frequency and amplitude variations 
    centered around the target f0/a0 (for positive examples).
    """
    f_min, f_max = f0 * (1 - freq_range_pct), f0 * (1 + freq_range_pct)
    a_min, a_max = a0 * (1 - amp_range_pct), a0 * (1 + amp_range_pct)
    
    # Frequency is generated within the tight margin (e.g., +/- 2% of f0)
    f = np.random.uniform(f_min, f_max)
    # Amplitude is generated within the variability margin
    a = np.random.uniform(a_min, a_max)
    
    phase = np.random.uniform(0, 2 * np.pi)
    return {"freq": float(f), "amp": float(a), "phase": float(phase)}


def corrupt_components_detune(components: List[Dict], freq_range_pct: float) -> List[Dict]:
    """
    NEW CORRUPTION MECHANISM: Frequency De-Tuning Test
    
    This function selects C components (1 <= C <= N_total) and shifts their frequency 
    to be OUTSIDE the positive match margin (freq_range_pct), while keeping the 
    component count, amplitude, and phase fixed.
    
    The shift is guaranteed to be between 1x and 2x the original margin away.
    """
    N_total = len(components)
    
    # 1. Determine C: number of components to de-tune (1 <= C <= N_total)
    C = random.randint(1, N_total)  
    indices_to_corrupt = random.sample(range(N_total), C)
    
    # Create a mutable copy of the component list
    corrupted_components = [c.copy() for c in components] 
    
    for idx in indices_to_corrupt:
        comp = corrupted_components[idx]
        f0 = comp['freq']
        
        # 1. Calculate the margin (M) based on the original positive variation
        M = f0 * freq_range_pct 
        
        # 2. Define the new frequency shift range: [M, 2M]
        shift_min = M
        shift_max = 2 * M
        
        # 3. Generate a random shift magnitude in the required range
        shift_amount = np.random.uniform(shift_min, shift_max)
        
        # 4. Randomly choose direction (up or down)
        if random.choice([True, False]): # Shift Up
            f_new = f0 + shift_amount
        else: # Shift Down
            f_new = f0 - shift_amount
            
        # 5. Clamp frequency to global limits [MIN_FREQ, MAX_FREQ]
        f_new = max(MIN_FREQ, min(MAX_FREQ, f_new))
        
        # Update component frequency (amp and phase remain the same)
        comp['freq'] = float(f_new)

    return corrupted_components


def synthesize_noisy_signal(components: List[Dict], duration=DURATION, sample_rate=SAMPLE_RATE) -> np.ndarray:
    """
    Synthesizes the time-series signal from the component parameters and adds
    White Gaussian Noise with magnitude NOISE_MAGNITUDE.
    """
    num_samples = int(duration * sample_rate)
    t = np.linspace(0, duration, num_samples, endpoint=False)
    
    output_signal_clean = np.zeros_like(t, dtype=np.float32)

    # 1. Generate Clean Signal (Sum of Sine Waves)
    for comp in components:
        freq = comp['freq']
        amp = comp['amp']
        phase = comp['phase']
        
        if amp > 0.0:
            signal_component = amp * np.sin(2 * np.pi * freq * t + phase)
            output_signal_clean += signal_component

    # 2. Add White Gaussian Noise (CRITICAL ADDITION)
    noise = np.random.normal(0.0, NOISE_MAGNITUDE, size=output_signal_clean.shape).astype(np.float32)
    
    output_signal_noisy = output_signal_clean + noise
    
    return output_signal_noisy


# -------------------------------------------
# Dataset generation (Saves Parameters only)
# -------------------------------------------

def generate_dataset_params(
    output_folder,
    n_samples,
    base_freqs,
    base_amps,
    freq_range_pct=DEFAULT_FREQ_RANGE_PCT,
    amp_range_pct=DEFAULT_AMP_RANGE_PCT
):
    """
    Generate dataset storing only parameters for positive and negative examples.
    The component list length is fixed and equals len(base_freqs).
    """
    pos_folder = os.path.join(output_folder, "positive")
    neg_folder = os.path.join(output_folder, "negative")
    os.makedirs(pos_folder, exist_ok=True)
    os.makedirs(neg_folder, exist_ok=True)

    # Generate one set of components for each signal pair
    for i in range(1, n_samples + 1):
        # --- POSITIVE EXAMPLE ---
        # Components are generated based on the specific base_freqs within a tight margin
        pos_components = [generate_component(f0, a0, freq_range_pct, amp_range_pct)
                          for f0, a0 in zip(base_freqs, base_amps)]
        
        # 1. Save Positive (Fixed Length)
        pos_path = os.path.join(pos_folder, f"signal_{i:04d}.json")
        with open(pos_path, "w") as f:
            # Note: No padding needed, list length is fixed at 25
            json.dump(pos_components, f, indent=2)

        # --- NEGATIVE EXAMPLE (Targeted De-Tuning) ---
        neg_components = corrupt_components_detune(pos_components, freq_range_pct)
        
        # 2. Save Negative (Fixed Length)
        neg_path = os.path.join(neg_folder, f"signal_{i:04d}.json")
        with open(neg_path, "w") as f:
            # Note: No padding needed, list length is fixed at 25
            json.dump(neg_components, f, indent=2)

    print(f"✅ Generated {n_samples} positive & negative parameter files in '{output_folder}'.")
    print(f"✅ All component lists have a fixed length of {len(base_freqs)}.")


# -------------------------------------------
# Example usage
# -------------------------------------------

if __name__ == "__main__":
    # --- FINAL COMPLEX CONFIGURATION: 25 components, 3 Sparse Clusters ---
    base_freqs = [
        # Cluster 1 (Low-Band) - 9 Frequencies
        515, 595, 680, 805, 950, 1110, 1260, 1385, 1490, 
        # Gap
        3100, 3600,
        # Cluster 2 (Mid-Band) - 8 Frequencies
        5600, 6050, 6420, 6980, 7550, 7880, 8150, 8490, 
        # Gap
        12150, 12800, 13900, 
        # Cluster 3 (High-Band) - 5 Frequencies
        18200, 19100, 19990
    ] 
    
    # Corresponding random amplitudes for variability (25 elements)
    base_amps = [
        0.85, 0.33, 0.92, 0.51, 0.76, 0.22, 0.98, 0.45,
        0.61, 0.88, 0.19, 0.70, 0.59, 0.94, 0.29, 0.73, 0.15,
        0.66, 0.89, 0.41, 0.99, 0.36, 0.82, 0.54, 0.17
    ] 
    
    print(f"ℹ️ Noise Magnitude: {NOISE_MAGNITUDE}.")
    print(f"ℹ️ Component List Length is fixed at: {len(base_freqs)}.")
    print(f"ℹ️ Positive Freq Variation: +/- {DEFAULT_FREQ_RANGE_PCT*100:.1f}%")
    print("ℹ️ Negative mechanism: Targeted frequency shift (De-Tuning Test).")
    
    # 1. Generate the parameter files 
    generate_dataset_params(
        output_folder="dataset_params",
        n_samples=20000, 
        base_freqs=base_freqs,
        base_amps=base_amps,
        freq_range_pct=DEFAULT_FREQ_RANGE_PCT,
        amp_range_pct=DEFAULT_AMP_RANGE_PCT
    )

    # 2. Demonstration of the new NOISY signal synthesis functionality
    print("\n--- Demonstration of Corrupted Component Lists ---")
    
    # Generate one set of positive parameters (fixed length 25)
    example_pos_components = [generate_component(f0, a0, DEFAULT_FREQ_RANGE_PCT, DEFAULT_AMP_RANGE_PCT)
                              for f0, a0 in zip(base_freqs, base_amps)]
    
    # Generate a corresponding negative list using the new logic
    example_neg_components = corrupt_components_detune(example_pos_components, DEFAULT_FREQ_RANGE_PCT)

    f0_target = base_freqs[0] # e.g., 515 Hz
    f_pos_actual = example_pos_components[0]['freq']
    f_neg_actual = example_neg_components[0]['freq']

    print(f"Target Base Freq (f0): {f0_target:.2f} Hz")
    print(f"Positive Actual Freq: {f_pos_actual:.2f} Hz (within +/- {DEFAULT_FREQ_RANGE_PCT*100:.1f}%)")
    print(f"Negative Actual Freq: {f_neg_actual:.2f} Hz (Corrupted, outside range)")

    # Synthesize the noisy signal (just for completeness)
    noisy_signal_array = synthesize_noisy_signal(example_pos_components)
    print(f"\nSynthesized noisy signal array of shape: {noisy_signal_array.shape}")

ℹ️ Noise Magnitude: 2.0.
ℹ️ Component List Length is fixed at: 25.
ℹ️ Positive Freq Variation: +/- 10.0%
ℹ️ Negative mechanism: Targeted frequency shift (De-Tuning Test).
✅ Generated 20000 positive & negative parameter files in 'dataset_params'.
✅ All component lists have a fixed length of 25.

--- Demonstration of Corrupted Component Lists ---
Target Base Freq (f0): 515.00 Hz
Positive Actual Freq: 537.71 Hz (within +/- 10.0%)
Negative Actual Freq: 500.00 Hz (Corrupted, outside range)

Synthesized noisy signal array of shape: (8000,)
